# Local UITI_VANO Interpretability

Notebook-first workflow for steps 1-3: circuit/date selection, deterministic critical-point detection, structured context, and optional LLM interpretation.

## 1. Setup and parameters

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

# 1. Dynamically locate the project root by finding the 'src' directory
ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

# 2. Add 'src' to sys.path so Python can find project packages
src_path = str(ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# 3. Load environment variables (.env tiene prioridad; si no existe, usa .env.example)
if load_dotenv:
    _env_file = ROOT / ".env" if (ROOT / ".env").exists() else ROOT / ".env.example"
    load_dotenv(_env_file)
    print(f"Config cargada desde: {_env_file.name}")

# 4. Import project modules
from chec_local_interpreter.attribution import enrich_critical_points
from chec_local_interpreter.config import CriticalityThresholds, DEFAULT_DATA_PATH, MAX_CRITICAL_POINTS as DEFAULT_MAX_CRITICAL_POINTS
from chec_local_interpreter.context_builder import build_context_package, critical_points_frame, save_json_artifact
from chec_local_interpreter.critical_points import build_daily_series, compute_daily_features, detect_critical_periods, detect_point_reasons, rank_critical_points
from chec_local_interpreter.data_loader import available_circuits, dataset_summary, filter_events, load_dataset, resolve_columns
from chec_local_interpreter.llm_client import call_llm, save_cot_html_graph
from chec_local_interpreter.llm_contracts import PROMPT_VERSION, load_output_schema, render_prompt, save_prompt_artifact
from chec_local_interpreter.llm_skills import assemble_skill_bundle, list_available_skills, verify_required_skills
from chec_local_interpreter.llm_validation import save_invalid_output, validate_llm_response
from chec_local_interpreter.plotting import save_uiti_vano_plot, plot_interactive_critical_points

# =======================================================
# Configuración común elegida por el usuario
# =======================================================
DATA_PATH = str(ROOT / "data/Indicadores_vano_v3.csv")
SELECTED_CIRCUITOS = ["CHA23L18"]
START_DATE = None  # Ej.: "2025-11-01 00:00:01"; None usa el inicio disponible.
END_DATE = None    # Ej.: "2026-04-30 21:53:40"; None usa el fin disponible.
MAX_CRITICAL_POINTS = 5
OUTPUT_DIR = Path("outputs")
CALL_LLM = True
_configured_provider = os.getenv("LLM_PROVIDER")
if _configured_provider:
    LLM_PROVIDER = _configured_provider
elif os.getenv("GOOGLE_API_KEY", "").strip():
    LLM_PROVIDER = "google"
elif os.getenv("OPENAI_API_KEY", "").strip():
    LLM_PROVIDER = "openai"
else:
    LLM_PROVIDER = "ollama"

if LLM_PROVIDER.lower() == "ollama":
    LLM_MODEL = os.getenv("LLM_MODEL") or os.getenv("OLLAMA_MODEL", "deepseek-r1:32b")
elif LLM_PROVIDER.lower() == "openai":
    LLM_MODEL = os.getenv("LLM_MODEL", "gpt-4.1-mini")
else:
    LLM_MODEL = os.getenv("LLM_MODEL", "gemini-2.5-flash-lite")

# Fechas para los escenarios de inferencia. None usa los días de critical_points.
FECHAS_INTERES_USUARIO = None

# Parámetros MGCECDL / SHAP comunes a 8.1-8.7
TOP_N_VANOS = 97
TOP_K_VARS = 20
SHAP_BACKGROUND_SIZE = 40
SHAP_NSAMPLES = 80
SHAP_BATCH_SIZE = 512
SHAP_RANDOM_STATE = 42
FILTRO_UITI_MAX = None
VENTANA_CLIMATICA_HORAS = 12
VARIABLES_SELECCION_PATH = ROOT / "data" / "Variables_seleccion.xlsx"
OPTUNA_STUDY_PATH = ROOT / "data" / "optuna" / "mgcecdl_classification_feature_attention_params.journal"


def _modelo_mas_reciente(model_dir: Path, base_name: str) -> Path:
    stem = Path(base_name).stem
    suffix = Path(base_name).suffix
    candidates = sorted(model_dir.glob(f"{stem}*{suffix}"))
    if not candidates:
        raise FileNotFoundError(f"No encontrado: {base_name} en {model_dir}")
    selected = candidates[-1]
    if len(candidates) > 1:
        print(f"{len(candidates)} versiones de {base_name}. Usando: {selected.name}")
    return selected


MODEL_PATH = _modelo_mas_reciente(ROOT / "data" / "models", "mgcecdl_classifier_best.zip")


## 2. Load data

In [ ]:
import pandas as pd
raw_df = load_dataset(DATA_PATH)
column_resolution = resolve_columns(raw_df)
summary = dataset_summary(raw_df)
available = available_circuits(raw_df)
selected_label = SELECTED_CIRCUITOS if SELECTED_CIRCUITOS else "all circuits"
print(f"Dataset shape: {summary['shape']}")
print(f"Date range: {summary['date_min']} to {summary['date_max']}")
print(f"Available circuits: {summary['available_circuits_count']}")
print(f"Selected circuits parameter: {selected_label}")
print(f"Unavailable optional columns: {len(column_resolution.unavailable_optional)}")


In [ ]:
raw_df.head()

## 2.1 Circuit Histograms: number of events and UITI

In [ ]:
from chec_local_interpreter.plotting import plot_interactive_circuit_events

# =======================================================
# EJEMPLOS DE USO:
# =======================================================

# 1. Utilizar todo el intervalo de datos (sin filtrar)
fig = plot_interactive_circuit_events(raw_df)
fig.show()

# 2. Utilizar un intervalo de fechas específico
#fig = plot_interactive_circuit_events(raw_df, start_date='2025-11-01 00:00:01', end_date='2025-12-01 00:00:01')
#fig.show()


In [ ]:
from chec_local_interpreter.plotting import plot_interactive_uiti_vano_sums

# =======================================================
# EJEMPLOS DE USO:
# =======================================================

# 1. Utilizar todo el intervalo de datos (sin filtrar)
fig_sums = plot_interactive_uiti_vano_sums(raw_df)
fig_sums.show()

# 2. Utilizar un intervalo de fechas específico
# fig_sums = plot_interactive_uiti_vano_sums(raw_df, start_date='2025-11-01 00:00:01', end_date='2025-12-01 00:00:01')
# fig_sums.show()


In [ ]:
from chec_local_interpreter.plotting import run_kmeans, plot_interactive_circuit_clustering

# =======================================================
# EJEMPLOS DE USO:
# =======================================================

# Lista de circuitos a destacar (opcional)
circuitos_destacados = [
     'CHA23L18', 'CHA23L14', 'BOA23L13', 'VBO23L14', 
     'BUM23L12', 'VBO23L15', 'SIO23L15', 'BOA23L12', 
     'AMA23L12', 'QHI23L13', 'IRR23L13', 'RIO23L14', 'DON23L12','DON23L13'
 ]

# 1. Utilizar todo el intervalo de datos con los destacados:
fig_cluster = plot_interactive_circuit_clustering(raw_df, highlighted_circuits=circuitos_destacados)
fig_cluster.show()

# 2. Utilizar un intervalo de fechas y sin circuitos destacados:
# fig_cluster2 = plot_interactive_circuit_clustering(raw_df, start_date='2025-11-01 00:00:01', end_date='2025-12-01 00:00:01')
# fig_cluster2.show()


## 2.2 GIS MAP

In [ ]:
from chec_local_interpreter.plotting import plot_circuit_map_plotly

# =================================================================
# Ejemplo de uso:
# =================================================================
plot_circuit_map_plotly(
    df=raw_df, 
    circuito_name='DON23L13', 
    date_range=None, 
    color_target='number_of_events'
)




In [ ]:
plot_circuit_map_plotly(
    df=raw_df, 
    circuito_name='DON23L13', 
    date_range=None, 
    color_target='UITI_VANO_sum'
)

## 3. Filter dataset

In [ ]:
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = str((ROOT / DATA_PATH).resolve() if not Path(DATA_PATH).is_absolute() else Path(DATA_PATH))
DATA_PATH


In [ ]:
events_df = filter_events(
    raw_df,
    selected_circuitos=SELECTED_CIRCUITOS,
    start_date=START_DATE,
    end_date=END_DATE,
)
selected_circuitos_effective = SELECTED_CIRCUITOS or sorted(events_df['CIRCUITO'].dropna().astype(str).unique().tolist())
start_effective = START_DATE or (events_df['fecha_dia'].min().date().isoformat() if not events_df.empty else None)
end_effective = END_DATE or (events_df['fecha_dia'].max().date().isoformat() if not events_df.empty else None)
print(f"Filtered rows: {len(events_df)}")
print(f"Effective circuits: {selected_circuitos_effective[:10]}{' ...' if len(selected_circuitos_effective) > 10 else ''}")
print(f"Effective window: {start_effective} to {end_effective}")

In [ ]:
events_df.head()

## 4. Build daily UITI_VANO series

In [ ]:
daily_df = build_daily_series(events_df)
display(daily_df.head())
print(f"Daily rows: {len(daily_df)}")
print(f"Total UITI_VANO: {daily_df['UITI_VANO'].sum() if not daily_df.empty else 0}")

In [ ]:
daily_df['event_count'].sum()

## 5. Detect relevant/critical points

In [ ]:
thresholds = CriticalityThresholds(max_points=MAX_CRITICAL_POINTS)
feature_df = compute_daily_features(daily_df, metric="UITI_VANO")
reasons = detect_point_reasons(feature_df, thresholds=thresholds, metric="UITI_VANO")
critical_points = rank_critical_points(feature_df, reasons, max_points=MAX_CRITICAL_POINTS, metric="UITI_VANO")
critical_periods = detect_critical_periods(feature_df, thresholds=thresholds, metric="UITI_VANO")
display(critical_points_frame(critical_points))
critical_periods

## 6. Enrich critical points

In [ ]:
critical_points = enrich_critical_points(events_df, critical_points)
print(f"Enriched critical points: {len(critical_points)}")
critical_points[:1]

In [ ]:
fig_critical = plot_interactive_critical_points(
    daily_df,
    critical_points,
    selected_circuitos=selected_circuitos_effective,
    start_date=start_effective,
    end_date=end_effective,
)
fig_critical.show()

## 7. Build structured context JSON

In [ ]:
context_package = build_context_package(
    raw_df=raw_df,
    events_df=events_df,
    daily_df=daily_df,
    critical_points=critical_points,
    critical_periods=critical_periods,
    selected_circuitos=selected_circuitos_effective,
    start_date=start_effective,
    end_date=end_effective,
)
print(json.dumps({
    'analysis_name': context_package['analysis_name'],
    'selected_context': context_package['selected_context'],
    'summary': context_package['summary'],
    'critical_point_count': len(context_package['critical_points']),
}, ensure_ascii=False, indent=2))

## 8. Inferencias MGCECDL — análisis de variables por circuito

Escenarios de atribución usando Kernel SHAP sobre el modelo MGCECDL de inferencia:

1. **Top-97 por UITI_VANO** — período completo.
2. **Top-97 por frecuencia** — período completo.
3. **Top-97 por UITI_VANO** — puntos críticos derivados automáticamente.
4. **Top-97 por frecuencia** — puntos críticos derivados automáticamente.

Cada escenario conserva el grafo interactivo estimado por reconstrucción MGCECDL + RBF y lo entrega al agente de inferencia.

In [ ]:
# Conexión con la configuración común y con los puntos críticos detectados.
CIRCUITO_INTERES = selected_circuitos_effective[0]
FECHA_INICIO = str(start_effective)
FECHA_FIN = str(end_effective)
FECHAS_INTERES = (
    list(FECHAS_INTERES_USUARIO)
    if FECHAS_INTERES_USUARIO is not None
    else [cp["fecha_dia"] for cp in critical_points]
)

print(f"Circuito       : {CIRCUITO_INTERES}")
print(f"Período        : {FECHA_INICIO}  →  {FECHA_FIN}")
print(f"Fechas críticas: {FECHAS_INTERES}")

In [ ]:
# Rutas requeridas por el flujo MGCECDL.
for _p in [VARIABLES_SELECCION_PATH, MODEL_PATH]:
    if not _p.exists():
        raise FileNotFoundError(f"No encontrado: {_p}")

print(f"Modelo MGCECDL : {MODEL_PATH}")
print(f"Top-N vanos    : {TOP_N_VANOS}")
print(f"Top-K variables: {TOP_K_VARS}")
print(f"SHAP background: {SHAP_BACKGROUND_SIZE}")
print(f"SHAP nsamples  : {SHAP_NSAMPLES}")
print(f"Filtro UITI max: {FILTRO_UITI_MAX}")
print(f"Ventana clima  : {VENTANA_CLIMATICA_HORAS} horas")

### 8.1 Carga, preprocesamiento y escalado MGCECDL

`procesar_dataset_completo` construye la matriz de features; luego se aplica el mismo escalado min-max usado por el entrenamiento MGCECDL antes de filtrar circuito y período.

In [ ]:
import io
import warnings
from contextlib import redirect_stdout

import numpy as np
from chec_impacto.data import preparar_splits_estratificados, procesar_dataset_completo
from chec_impacto.training import escalar_features_minmax_mgcecdl, resolve_training_device

DEVICE = resolve_training_device("auto")
print(f"Usando device: {DEVICE}")

with redirect_stdout(io.StringIO()):
    datos_inferencia = procesar_dataset_completo(
        path_clima=DATA_PATH,
        path_variables_seleccion=VARIABLES_SELECCION_PATH,
        use_sampling=False,
        min_samples_per_codigo=5,
        target="UITI_VANO",
        filtro_uiti_max=FILTRO_UITI_MAX,
        ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
    )

X_full_raw = np.asarray(datos_inferencia["X"], dtype=np.float32)
features = list(datos_inferencia["features"])
base_full = datos_inferencia["df_original_copy"].copy().reset_index(drop=True)

splits_clf = escalar_features_minmax_mgcecdl(
    preparar_splits_estratificados(
        X_full_raw,
        datos_inferencia["y"],
        modo="clasificacion",
        random_state=SHAP_RANDOM_STATE,
    )
)
feature_scaler = splits_clf["feature_scaler"]
X_full = feature_scaler.transform(X_full_raw).astype(np.float32)

fechas_col_inf = pd.to_datetime(base_full["FECHA"], errors="coerce")
mascara_inf = (
    base_full["CIRCUITO"].astype(str).str.strip().eq(CIRCUITO_INTERES)
    & fechas_col_inf.ge(pd.Timestamp(FECHA_INICIO))
    & fechas_col_inf.le(pd.Timestamp(FECHA_FIN))
)
if not mascara_inf.any():
    raise ValueError(f"Sin datos de inferencia para {CIRCUITO_INTERES} entre {FECHA_INICIO} y {FECHA_FIN}.")

X_inf = X_full[mascara_inf.to_numpy()]
base_inf = base_full[mascara_inf].copy().reset_index(drop=True)
base_inf["_FECHA_DIA"] = fechas_col_inf[mascara_inf].dt.strftime("%Y-%m-%d").values

print(f"Eventos      : {len(X_inf):,}")
print(f"Vanos únicos : {base_inf['FID_VANO'].nunique():,}")
print(f"Features     : {len(features)}")

### 8.2 Modelo MGCECDL y Kernel SHAP

In [ ]:
from chec_impacto.training import cargar_estudio_optuna_mgcecdl, cargar_modelo_mgcecdl, predict_classification

with warnings.catch_warnings(), redirect_stdout(io.StringIO()):
    warnings.simplefilter("ignore")
    modelo_inferencia = cargar_modelo_mgcecdl(str(MODEL_PATH), device=DEVICE)

if OPTUNA_STUDY_PATH.exists():
    study_clf = cargar_estudio_optuna_mgcecdl(OPTUNA_STUDY_PATH, "clasificacion")
    GRAFO_ESTIMADO_RBF_SIGMA = float(study_clf.best_params.get("rbf_sigma", 1.0))
else:
    warnings.warn(
        f"No existe {OPTUNA_STUDY_PATH}; se usará rbf_sigma=1.0 para el grafo estimado.",
        RuntimeWarning,
    )
    GRAFO_ESTIMADO_RBF_SIGMA = 1.0

print(f"Modelo: {type(modelo_inferencia).__name__} | {MODEL_PATH.name}")
print(f"rbf_sigma para grafo estimado: {GRAFO_ESTIMADO_RBF_SIGMA}")

In [ ]:
from chec_impacto.interpretability.circuit_analysis import KernelShapTopVarsExtractor


class MGCECDLClassifierShapAdapter:
    """Adaptador predict_proba para reutilizar Kernel SHAP con MGCECDL."""

    def __init__(self, model, device):
        self.model = model
        self.device = device

    def predict_proba(self, values):
        values = np.asarray(values, dtype=np.float32)
        if values.ndim == 1:
            values = values.reshape(1, -1)
        return np.asarray(
            predict_classification(self.model, values, device=self.device)["fused_probs"],
            dtype=np.float64,
        )


modelo_shap = MGCECDLClassifierShapAdapter(modelo_inferencia, DEVICE)
shap_extractor = KernelShapTopVarsExtractor(
    model=modelo_shap,
    X=X_inf,
    features=features,
    top_k=TOP_K_VARS,
    background_size=SHAP_BACKGROUND_SIZE,
    nsamples=SHAP_NSAMPLES,
    batch_size=SHAP_BATCH_SIZE,
    random_state=SHAP_RANDOM_STATE,
)
print(
    f"Kernel SHAP configurado | fondo={shap_extractor.n_background} "
    f"| nsamples={SHAP_NSAMPLES} | batch={SHAP_BATCH_SIZE}"
)

### 8.3 Configuración de escenarios, modos y grafos

In [ ]:
from chec_impacto.interpretability.circuit_analysis import (
    agrupar_por_vano,
    construir_contexto_escenario_inferencia,
    construir_contexto_inferencia,
    construir_modos_chec,
    construir_prompt_inferencia,
    graficar_barras_y_radar,
    validar_respuesta_inferencia,
)

MODOS = construir_modos_chec(features, VARIABLES_SELECCION_PATH)
tabla_periodo_inf = agrupar_por_vano(base_inf)
n_top = min(TOP_N_VANOS, len(tabla_periodo_inf))
inference_results = {}

GRAPH_OUTPUT_DIR = ROOT / "data" / "results_mgcecdl" / "grafos_interactivos"
GRAPH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Modos        : {len(MODOS)}")
print(f"Vanos período: {len(tabla_periodo_inf)} | Top-N efectivo: {n_top}")
print(f"Grafos HTML  : {GRAPH_OUTPUT_DIR}")

In [ ]:
def ejecutar_escenario_inferencia(
    key,
    nombre,
    criterio,
    tabla_top,
    eventos_escenario,
    graph_output_name,
    fechas_interes=None,
):
    resultado = graficar_barras_y_radar(
        eventos_escenario,
        nombre,
        circuito=CIRCUITO_INTERES,
        features=features,
        modos=MODOS,
        shap_extractor=shap_extractor,
        top_k=TOP_K_VARS,
        graph_source="estimated",
        estimated_graph_model=modelo_inferencia,
        X_model=X_inf,
        estimated_graph_rbf_sigma=GRAFO_ESTIMADO_RBF_SIGMA,
        estimated_graph_device=DEVICE,
        estimated_graph_batch_size=SHAP_BATCH_SIZE,
        graph_output_dir=GRAPH_OUTPUT_DIR,
        graph_output_name=graph_output_name,
    )
    resultado["contexto"] = construir_contexto_escenario_inferencia(
        nombre=nombre,
        criterio=criterio,
        resultado=resultado,
        tabla_top=tabla_top,
        modos=MODOS,
        top_k=TOP_K_VARS,
        fechas_interes=fechas_interes,
        ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
    )
    inference_results[key] = resultado
    return resultado

### 8.4 Top-97 por UITI_VANO — período completo

In [ ]:
tabla_top_uiti = (
    tabla_periodo_inf
    .sort_values("UITI_VANO_PROM", ascending=False, kind="stable")
    .head(n_top)
    .reset_index(drop=True)
)
ids_top_uiti = tabla_top_uiti["FID_VANO"].tolist()
base_top_uiti = base_inf[base_inf["FID_VANO"].isin(ids_top_uiti)].copy()

resultado_top_uiti = ejecutar_escenario_inferencia(
    key="top_uiti_periodo",
    nombre=f"Top-{n_top} por UITI_VANO — período completo",
    criterio="ordenar por UITI_VANO_PROM descendente en el período completo",
    tabla_top=tabla_top_uiti,
    eventos_escenario=base_top_uiti,
    graph_output_name="top_uiti_periodo.html",
)

### 8.5 Top-97 por frecuencia — período completo

In [ ]:
tabla_top_frecuencia = (
    tabla_periodo_inf
    .sort_values(["N_APARICIONES", "UITI_VANO_PROM"], ascending=[False, False], kind="stable")
    .head(n_top)
    .reset_index(drop=True)
)
ids_top_frecuencia = tabla_top_frecuencia["FID_VANO"].tolist()
base_top_frecuencia = base_inf[base_inf["FID_VANO"].isin(ids_top_frecuencia)].copy()

resultado_top_frecuencia = ejecutar_escenario_inferencia(
    key="top_frecuencia_periodo",
    nombre=f"Top-{n_top} por frecuencia — período completo",
    criterio="ordenar por N_APARICIONES descendente y usar UITI_VANO_PROM como desempate",
    tabla_top=tabla_top_frecuencia,
    eventos_escenario=base_top_frecuencia,
    graph_output_name="top_frecuencia_periodo.html",
)

### 8.6 Top-97 — puntos críticos

Las fechas se derivan automáticamente de `critical_points`. Se calculan dos lecturas: gravedad (`UITI_VANO_PROM`) y número de eventos (`N_APARICIONES`).

In [ ]:
base_fechas_inf = base_inf[base_inf["_FECHA_DIA"].isin(FECHAS_INTERES)].copy()
if base_fechas_inf.empty:
    raise ValueError(f"No hay eventos de inferencia para las fechas críticas: {FECHAS_INTERES}")

tabla_fechas_inf = agrupar_por_vano(base_fechas_inf)
n_top_fechas = min(TOP_N_VANOS, len(tabla_fechas_inf))
fechas_label = ", ".join(FECHAS_INTERES)

# Gravedad en puntos críticos
tabla_top_fechas_uiti = (
    tabla_fechas_inf
    .sort_values("UITI_VANO_PROM", ascending=False, kind="stable")
    .head(n_top_fechas)
    .reset_index(drop=True)
)
ids_top_fechas_uiti = tabla_top_fechas_uiti["FID_VANO"].tolist()
base_top_fechas_uiti = base_fechas_inf[base_fechas_inf["FID_VANO"].isin(ids_top_fechas_uiti)].copy()

resultado_top_fechas_uiti = ejecutar_escenario_inferencia(
    key="top_uiti_puntos_criticos",
    nombre=f"Top-{n_top_fechas} por UITI_VANO — puntos críticos ({fechas_label})",
    criterio="filtrar fechas críticas y ordenar por UITI_VANO_PROM descendente",
    tabla_top=tabla_top_fechas_uiti,
    eventos_escenario=base_top_fechas_uiti,
    graph_output_name="top_uiti_fechas.html",
    fechas_interes=FECHAS_INTERES,
)

# Número de eventos en puntos críticos
tabla_top_fechas_frecuencia = (
    tabla_fechas_inf
    .sort_values(["N_APARICIONES", "UITI_VANO_PROM"], ascending=[False, False], kind="stable")
    .head(n_top_fechas)
    .reset_index(drop=True)
)
ids_top_fechas_frecuencia = tabla_top_fechas_frecuencia["FID_VANO"].tolist()
base_top_fechas_frecuencia = base_fechas_inf[base_fechas_inf["FID_VANO"].isin(ids_top_fechas_frecuencia)].copy()

resultado_top_fechas_frecuencia = ejecutar_escenario_inferencia(
    key="top_frecuencia_puntos_criticos",
    nombre=f"Top-{n_top_fechas} por frecuencia — puntos críticos ({fechas_label})",
    criterio="filtrar fechas críticas y ordenar por N_APARICIONES descendente, usando UITI_VANO_PROM como desempate",
    tabla_top=tabla_top_fechas_frecuencia,
    eventos_escenario=base_top_fechas_frecuencia,
    graph_output_name="top_frecuencia_fechas.html",
    fechas_interes=FECHAS_INTERES,
)

### 8.7 Agente de inferencias — discusión estructurada y grafos para HTML

In [ ]:
inference_context_package = construir_contexto_inferencia(
    circuito_interes=CIRCUITO_INTERES,
    fecha_inicio=FECHA_INICIO,
    fecha_fin=FECHA_FIN,
    fechas_interes=FECHAS_INTERES,
    top_n_vanos=TOP_N_VANOS,
    top_k_vars=TOP_K_VARS,
    filtro_uiti_max=FILTRO_UITI_MAX,
    ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
    features=features,
    base=base_inf,
    escenarios=[resultado["contexto"] for resultado in inference_results.values()],
    modelo=type(modelo_inferencia).__name__,
    estimated_graph_rbf_sigma=GRAFO_ESTIMADO_RBF_SIGMA,
)

# El agente base también recibe la síntesis de inferencia y las rutas HTML de grafos.
context_package["inference_context"] = inference_context_package

inference_context_path = save_json_artifact(
    inference_context_package,
    OUTPUT_DIR / f"inference_structured_context_{timestamp}.json",
)

missing_inference_skills = verify_required_skills(profile="inferencia")
if missing_inference_skills:
    raise FileNotFoundError(f"Missing required inference LLM skills: {missing_inference_skills}")

inference_skill_bundle = assemble_skill_bundle(profile="inferencia")
inference_prompt = construir_prompt_inferencia(inference_context_package, inference_skill_bundle)
inference_prompt_path = save_prompt_artifact(inference_prompt, OUTPUT_DIR / f"inference_llm_prompt_{timestamp}.md")

print(f"Inference context saved: {inference_context_path}")
print(f"Inference prompt saved : {inference_prompt_path}")
print(f"Inference skills loaded: {list_available_skills(profile='inferencia')}")
print("Grafos entregados al agente:")
for graph_info in inference_context_package["graph_html_paths"]:
    print(f"- {graph_info['escenario']}: {graph_info['path']}")

### 8.8 Llamada y validación del agente de inferencia

In [ ]:
inference_validation_data = None
inference_llm_result = call_llm(
    inference_prompt,
    provider=LLM_PROVIDER,
    model=LLM_MODEL,
    call_enabled=CALL_LLM,
    display_progress=True,
    display_content=False,
)
print(inference_llm_result.message)

if inference_llm_result.output_text:
    inference_validation = validar_respuesta_inferencia(inference_llm_result.output_text, inference_context_package)
    if inference_validation["ok"]:
        inference_validation_data = inference_validation["data"]
        inference_analysis_path = save_json_artifact(
            inference_validation_data,
            OUTPUT_DIR / f"inference_llm_analysis_{timestamp}.json",
        )
        print(f"Inference LLM analysis saved: {inference_analysis_path}")
    else:
        raw_path, errors_path = save_invalid_output(
            inference_llm_result.output_text,
            inference_validation["errors"],
            OUTPUT_DIR,
            f"inference_{timestamp}",
        )
        print("La salida del agente de inferencias no validó; se guardó como inválida y no se presentará como análisis final.")
        print(f"Raw: {raw_path}")
        print(f"Errors: {errors_path}")
else:
    print("Agente de inferencias no ejecutado o sin salida; el HTML incluirá gráficas y grafos sin discusión de ese agente.")

## 9. Save artifacts

In [ ]:
context_path = save_json_artifact(context_package, OUTPUT_DIR / f"structured_context_{timestamp}.json")
critical_csv_path = OUTPUT_DIR / f"critical_points_{timestamp}.csv"
critical_points_frame(critical_points).to_csv(critical_csv_path, index=False)
plot_path = save_uiti_vano_plot(
    daily_df,
    critical_points,
    selected_circuitos=selected_circuitos_effective,
    start_date=start_effective,
    end_date=end_effective,
    output_path=OUTPUT_DIR / f"uiti_vano_timeseries_{timestamp}.png",
)
print(context_path)
print(critical_csv_path)
print(plot_path)

## 10. LLM skills, contracts, and validation

In [ ]:
# El agente base usa la misma configuración de proveedor/modelo que el agente de inferencias.
# No se hace verificación previa ni descarga de modelos en esta celda: `call_llm` usará
# el proveedor configurado y reportará de forma clara si no hay credenciales o servicio.
print(f"CALL_LLM   : {CALL_LLM}")
print(f"LLM_PROVIDER: {LLM_PROVIDER}")
print(f"LLM_MODEL   : {LLM_MODEL}")


In [ ]:
missing_skills = verify_required_skills()
if missing_skills:
    raise FileNotFoundError(f"Missing required LLM skills: {missing_skills}")
skills = list_available_skills()
skill_bundle = assemble_skill_bundle()
output_schema = load_output_schema()
prompt = render_prompt(
    context_json=json.dumps(context_package, ensure_ascii=False),
    output_schema_json=json.dumps(output_schema, ensure_ascii=False),
    prompt_version=PROMPT_VERSION,
)
prompt_path = save_prompt_artifact(prompt, OUTPUT_DIR / f"llm_prompt_{timestamp}.md")
print(f"Loaded skills: {skills}")
print(f"Prompt saved: {prompt_path}")
print(f"Skill bundle characters: {len(skill_bundle)}")

In [ ]:
validation_data = None
last_base_llm_result = None
last_base_validation = None

base_prompt_attempts = [prompt]
for attempt_idx, current_prompt in enumerate(base_prompt_attempts, start=1):
    llm_result = call_llm(
        current_prompt,
        provider=LLM_PROVIDER,
        model=LLM_MODEL,
        call_enabled=CALL_LLM,
        display_progress=True,
    display_content=False,
    )
    last_base_llm_result = llm_result
    print(f"Intento agente base {attempt_idx}: {llm_result.message}")

    if not llm_result.output_text:
        continue

    validation = validate_llm_response(llm_result.output_text, context_package, output_schema)
    last_base_validation = validation
    if validation.ok and validation.data:
        validation_data = validation.data
        analysis_path = save_json_artifact(
            validation_data,
            OUTPUT_DIR / f"llm_analysis_{timestamp}.json",
        )
        print(f"LLM analysis JSON saved: {analysis_path}")
        break

    raw_path, errors_path = save_invalid_output(
        llm_result.output_text,
        validation.errors,
        OUTPUT_DIR,
        f"base_attempt_{attempt_idx}_{timestamp}",
    )
    print("La salida del agente base no validó; se guardó para revisión.")
    print(f"Raw: {raw_path}")
    print(f"Errors: {errors_path}")

    if attempt_idx == 1:
        repair_prompt = f"""
{prompt}

---

La respuesta anterior NO cumplió el contrato JSON o las reglas de validación.
Errores detectados:
{json.dumps(validation.errors, ensure_ascii=False, indent=2)}

Vuelve a responder desde cero con SOLO JSON válido, sin markdown, sin texto adicional,
respetando exactamente el schema entregado y usando únicamente fechas, critical_point_id,
variables y resúmenes presentes en el contexto.
"""
        base_prompt_attempts.append(repair_prompt)

if validation_data is None:
    message = "El agente base no produjo un JSON válido. No se generará el HTML."
    if last_base_validation is not None:
        message += f" Últimos errores: {last_base_validation.errors}"
    elif last_base_llm_result is not None:
        message += f" Último estado: {last_base_llm_result.message}"
    raise RuntimeError(message)


# 11 Html results visualization

In [ ]:
from chec_local_interpreter.plotting import render_llm_analysis

if not globals().get("validation_data"):
    raise RuntimeError("validation_data está vacío; no se genera HTML sin análisis base válido.")

html_path = render_llm_analysis(
    validation_data=validation_data,
    raw_df=raw_df,
    daily_df=daily_df,
    critical_points=critical_points,
    selected_circuitos=selected_circuitos_effective,
    start_date=start_effective,
    end_date=end_effective,
    llm_model=LLM_MODEL,
    llm_provider=LLM_PROVIDER,
    inference_results=globals().get("inference_results", {}),
    inference_analysis=globals().get("inference_validation_data", None),
)
print(
    f"\n>>> HTML Analysis Report guardado en la ruta de outputs: "
    f"{html_path}"
)